In [0]:
%run "../Includes/configuration"

In [0]:
%run "../Includes/common_functions"

In [0]:
dbutils.widgets.text("p_file_date", "2024-12-16")
v_file_date = dbutils.widgets.get("p_file_date")

In [0]:
movie_df = spark.read.table("movie_silver.movies") \
                    .filter(f"file_date = '{v_file_date}'")

In [0]:
movie_language_df = spark.read.table("movie_silver.movie_languages") \
                              .filter(f"file_date = '{v_file_date}'")

In [0]:
language_df = spark.read.table("movie_silver.languages")

In [0]:
movie_genre_df = spark.read.table("movie_silver.movie_genres") \
                           .filter(f"file_date = '{v_file_date}'")

In [0]:
genre_df = spark.read.table("movie_silver.genres")

In [0]:
from pyspark.sql.functions import col, year, current_timestamp, lit

In [0]:
# movie_language_name = movie_df \
#                      .join(movie_language_df,
#                            movie_df.movie_Id == movie_language_df.movie_Id,
#                            "inner") \
#                      .join(language_df,
#                            movie_language_df.language_Id == language_df.Language_Id,
#                            "inner") \
#                      .join(movie_genre_df,
#                            movie_df.movie_Id == movie_genre_df.movie_Id,
#                            "inner") \
#                      .join(genre_df,
#                            movie_genre_df.genre_Id == genre_df.genre_Id,
#                            "inner") \
#                      .select(movie_df.title,
#                              movie_df.duration_Time,
#                              movie_df.release_Date,
#                              movie_df.vote_Average,
#                              language_df.Language_Name,
#                              genre_df.genre_Name
#                              ) \
#                      .withColumn("created_date", lit(v_file_date)) \
#                      .filter(year(movie_df.release_Date) > 2000 ) \
#                      .orderBy(movie_df.release_Date.desc())

In [0]:
movie_language_name_df = movie_df \
                     .join(movie_language_df,
                           movie_df.movie_Id == movie_language_df.movie_Id,
                           "inner") \
                     .join(language_df,
                           movie_language_df.language_Id == language_df.Language_Id,
                           "inner") \
                     .join(movie_genre_df,
                           movie_df.movie_Id == movie_genre_df.movie_Id,
                           "inner") \
                     .join(genre_df,
                           movie_genre_df.genre_Id == genre_df.genre_Id,
                           "inner") \
                     .select(movie_df.movie_Id,
                             language_df.Language_Id,
                             genre_df.genre_Id,
                             movie_df.title,
                             movie_df.duration_Time,
                             movie_df.release_Date,
                             movie_df.vote_Average,
                             language_df.Language_Name,
                             genre_df.genre_Name
                             ) \
                     .withColumn("created_date", lit(v_file_date)) \
                     .filter(year(movie_df.release_Date) > 2000 ) \
                     .orderBy(movie_df.release_Date.desc())

In [0]:
display(movie_language_name_df)

movie_Id,Language_Id,genre_Id,title,duration_Time,release_Date,vote_Average,Language_Name,genre_Name,created_date
10317,24574,35,Our Brand Is Crisis,7002261,2015-01-01,null,English,Comedy,2024-12-16
10317,24574,18,Our Brand Is Crisis,7002261,2015-01-01,null,English,Drama,2024-12-16
189,24574,80,Sin City: A Dame to Kill For,102,2014-08-20,6.3,English,Crime,2024-12-16
189,24574,53,Sin City: A Dame to Kill For,102,2014-08-20,6.3,English,Thriller,2024-12-16
4258,24574,35,Scary Movie 5,86,2013-04-11,4.6,English,Comedy,2024-12-16
1930,24574,28,The Amazing Spider-Man,136,2012-06-27,6.5,English,Action,2024-12-16
1930,24574,14,The Amazing Spider-Man,136,2012-06-27,6.5,English,Fantasy,2024-12-16
1930,24574,12,The Amazing Spider-Man,136,2012-06-27,6.5,English,Adventure,2024-12-16
17578,24574,9648,The Adventures of Tintin,107,2011-10-25,6.7,English,Mystery,2024-12-16
17578,24574,16,The Adventures of Tintin,107,2011-10-25,6.7,English,Animation,2024-12-16


In [0]:
# overwrite_partition("movie_gold", "results_movie_genre_language", "created_date", v_file_date)

In [0]:
merge_delta_lake_3(movie_language_name_df, "movie_gold", "results_movie_genre_language", "movie_Id", "Language_Id", "genre_Id", "created_date")

In [0]:
# movie_language_name_df.write.mode("append").partitionBy("created_date").format("delta").saveAsTable("movie_gold.results_movie_genre_language")

In [0]:
display(spark.read.table("movie_gold.results_movie_genre_language"))

movie_Id,Language_Id,genre_Id,title,duration_Time,release_Date,vote_Average,Language_Name,genre_Name,created_date
10317,24574,35,Our Brand Is Crisis,7002261,2015-01-01,null,English,Comedy,2024-12-16
10317,24574,18,Our Brand Is Crisis,7002261,2015-01-01,null,English,Drama,2024-12-16
189,24574,80,Sin City: A Dame to Kill For,102,2014-08-20,6.3,English,Crime,2024-12-16
189,24574,53,Sin City: A Dame to Kill For,102,2014-08-20,6.3,English,Thriller,2024-12-16
4258,24574,35,Scary Movie 5,86,2013-04-11,4.6,English,Comedy,2024-12-16
1930,24574,28,The Amazing Spider-Man,136,2012-06-27,6.5,English,Action,2024-12-16
1930,24574,14,The Amazing Spider-Man,136,2012-06-27,6.5,English,Fantasy,2024-12-16
1930,24574,12,The Amazing Spider-Man,136,2012-06-27,6.5,English,Adventure,2024-12-16
17578,24574,9648,The Adventures of Tintin,107,2011-10-25,6.7,English,Mystery,2024-12-16
17578,24574,16,The Adventures of Tintin,107,2011-10-25,6.7,English,Animation,2024-12-16


In [0]:
%sql
SELECT created_date, COUNT(1)
FROM movie_gold.results_movie_genre_language
GROUP BY created_date;

created_date,count(1)
2024-12-16,5181
2024-12-23,1033
2024-12-30,912


In [0]:
%sql
/*aqui no leera la informacion desde el archivo de la cuenta de almacenamiento, sino desde la tabla creada en la base de datos movie_gold*, lo cual es lo mismo que el dataframe anterior*/

SELECT * FROM movie_gold.results_movie_genre_language;


movie_Id,Language_Id,genre_Id,title,duration_Time,release_Date,vote_Average,Language_Name,genre_Name,created_date
10317,24574,35,Our Brand Is Crisis,7002261,2015-01-01,null,English,Comedy,2024-12-16
10317,24574,18,Our Brand Is Crisis,7002261,2015-01-01,null,English,Drama,2024-12-16
189,24574,80,Sin City: A Dame to Kill For,102,2014-08-20,6.3,English,Crime,2024-12-16
189,24574,53,Sin City: A Dame to Kill For,102,2014-08-20,6.3,English,Thriller,2024-12-16
4258,24574,35,Scary Movie 5,86,2013-04-11,4.6,English,Comedy,2024-12-16
1930,24574,28,The Amazing Spider-Man,136,2012-06-27,6.5,English,Action,2024-12-16
1930,24574,14,The Amazing Spider-Man,136,2012-06-27,6.5,English,Fantasy,2024-12-16
1930,24574,12,The Amazing Spider-Man,136,2012-06-27,6.5,English,Adventure,2024-12-16
17578,24574,9648,The Adventures of Tintin,107,2011-10-25,6.7,English,Mystery,2024-12-16
17578,24574,16,The Adventures of Tintin,107,2011-10-25,6.7,English,Animation,2024-12-16


In [0]:
%sql
DESCRIBE EXTENDED movie_gold.results_movie_genre_language;

col_name,data_type,comment
movie_Id,int,null
Language_Id,int,null
genre_Id,int,null
title,string,null
duration_Time,int,null
release_Date,date,null
vote_Average,double,null
Language_Name,string,null
genre_Name,string,null
created_date,string,null
